### Parâmetros necessários
- Comprimento de Bjerrum:	$l_B$	m
- Comprimento de Debye:	$\kappa^{-1}$	m
- Volume excluído não eletrostático:	$\omega$ Adimensional
- Comprimento do segmento estatístico:	$b$	m
- Número de segmentos	$N$	Adimensional
- Fração de cargas positivas:	$f$	Adimensional
- Fração de cargas negativas:	$g$	Adimensional

**Parâmetro adimensional para o volume físico**
$$
\omega^* =
\omega+
\frac{4\pi l_B(f-g)^2}{b^3\kappa^2}
-
\frac{\pi l_B^2(f+g)^2}{b^3\kappa}
$$

**Comprimento de Bjerrum**
$$
l_B=\frac{e^2}{4\pi\varepsilon_0\varepsilon_r k_BT}
$$

**Comprimento de Debye**

Para uma solução com força iônica $I$, expressa em mol/L:
$$
\kappa^2 =
\frac{2e^2N_A(1000)I}
{\varepsilon_0\varepsilon_r k_BT}
$$

**Fator de expansão** <br>
Utilizado para calcular como a cadeia se expande ou se compacta em relação à sua conformação ideal.
$$
\alpha^5-\alpha^3 =
\frac{4}{3}
\left(\frac{3}{2\pi}\right)^{3/2}
N^{1/2}\omega^*
$$

### Implementação 

In [ ]:

import numpy as np

from scipy.constants import (
    elementary_charge as e,
    epsilon_0,
    Boltzmann as k_B,
    Avogadro as N_A
)


# Funções auxiliares

def bjerrum_length(T, epsilon_r):
    return e**2 / (
        4 * np.pi * epsilon_0 * epsilon_r * k_B * T
    )


def debye_length(T, epsilon_r, ionic_strength_M):
    kappa_sq = (
        2 * e**2 * N_A * 1000 * ionic_strength_M
        / (epsilon_0 * epsilon_r * k_B * T)
    )
    kappa = np.sqrt(kappa_sq)
    return 1 / kappa, kappa


# MODELO DE JOANNY (1999): ADSORÇÃO E INVERSÃO DE CARGA
def joanny_adsorption_layer(
    T=298.15,
    epsilon_r=78.5,          # Água
    ionic_strength_M=0.01,   # 10 mM (força iônica típica)
    sigma_surface_nm2,  
    b_nm,           
    N,               
    f                   
):
    """
    Calcula a carga sobrecompensada e a espessura de uma camada adsorvida
    de polieletrólito (PPy) sobre uma superfície carregada (MoS2),
    seguindo o modelo de campo médio de Joanny (1999).
    
    Retorna um dicionário com os resultados e o regime identificado.
    """
    
    # --- Conversão de unidades ---
    b = b_nm * 1e-9                # m
    sigma_nm2 = sigma_surface_nm2  # cargas/nm² (para output)
    sigma_m2 = sigma_nm2 * 1e18    # cargas/m²
    
    # --- Comprimentos eletrostáticos ---
    l_B = bjerrum_length(T, epsilon_r)
    lambda_D, kappa = debye_length(T, epsilon_r, ionic_strength_M)
    
    # --- Critério de regime ---
    # Joanny define a transição pela comparação entre a espessura da camada
    # e o comprimento de Debye. Usamos a espessura de uma cadeia ideal
    # (b*sqrt(N)) como escala de referência inicial.
    D_est = b * np.sqrt(N)
    kappa_D_est = kappa * D_est
    
    # O artigo de Joanny não fornece um critério numérico exato para kappa*D,
    # mas a física indica: kappa*D << 1 => baixa força iônica.
    # Usamos kappa*D_est < 1 como aproximação razoável.
    
    if kappa_D_est < 1.0:
        # REGIME DE BAIXA FORÇA IÔNICA (Screening Fraco)
        regime = "Baixa Força Iônica (Screening Fraco)"
        
        # Espessura da camada adsorvida: ordem da espessura de uma única
        # cadeia adsorvida. Joanny (1999) indica que D ~ R_ads.
        # para uma cadeia fracamente carregada adsorvida:
        # D ~ b * N^(1/2) * (f * l_B / b)^(1/3) (estimativa de escala)
        D = b * np.sqrt(N) * (f * l_B / b)**(1/3)
        
        # Carga sobrecompensada: proporcional a kappa (Joanny 1999)
        # sigma_over ~ sigma_surface * (1 + C * kappa * D)
        # O fator C é uma constante de ordem 1; usamos 0.5 como estimativa
        # conservadora para evitar superestimação.
        sigma_over_m2 = sigma_m2 * (1 + 0.5 * kappa * D)
        
    else:
        # REGIME DE ALTA FORÇA IÔNICA (Screening Forte)
        regime = "Alta Força Iônica (Screening Forte)"
        
        # interação eletrostática fortemente blindada
        # espessura reduzida pelo screening:
        # D ~ b * N^(1/2) / (kappa * b * N^(1/2))^(2/3)
        D = b * np.sqrt(N) / (kappa * b * np.sqrt(N))**(2/3)
        
        # Carga sobrecompensada: proporcional à carga nua da superfície.
        # Joanny (1999): sigma_over ~ sigma_surface
        sigma_over_m2 = sigma_m2
    
    # --- Conversão de volta para unidades convenientes ---
    D_nm = D * 1e9
    sigma_over_nm2 = sigma_over_m2 * 1e-18
    
    # --- Fator de sobrecompensação ---
    sobrecompensacao = sigma_over_nm2 / sigma_nm2
    
    return {
        "Regime": regime,
        "Carga Superfície (cargas/nm²)": sigma_nm2,
        "Carga Sobrecompensada (cargas/nm²)": sigma_over_nm2,
        "Fator de Sobrecompensação": sobrecompensacao,
        "Espessura da Camada (nm)": D_nm,
        "Comprimento de Debye (nm)": lambda_D * 1e9,
        "Bjerrum (nm)": l_B * 1e9,
        "kappa * D_est (critério)": kappa_D_est
    }

# Modelagem do LbL (PPy / MoS2)
def simular_multicamadas_lbl(
    n_camadas,
    T,
    epsilon_r,
    ionic_strength_M,
    sigma_inicial_nm2,
    b_nm_ppy,
    N_ppy,
    f_ppy,
    chains_adsorbed_nm2,
    sigma_mos2_layer_nm2,
    espessura_mos2_nm
):
    """
    Simulação simplificada da deposição alternada de PPy e MoS2.
    A carga superficial é assinada.
    PPy:
        Carga positiva.
    MoS2:
        Carga negativa.
    A deposição de PPy utiliza a estimativa de adsorção
    de polieletrólito.
    """

    resultados = []

    sigma_atual = sigma_inicial_nm2

    for i in range(n_camadas):
        # Determinação do material
        if i % 2 == 0:
            material = "PPy"
            sinal_material = +1
        else:
            material = "MoS2"
            sinal_material = -1

        sigma_antes = sigma_atual

        # ADSORÇÃO DO PPy

        if material == "PPy":

            res = joanny_adsorption_layer(
                T=T,
                epsilon_r=epsilon_r,
                ionic_strength_M=ionic_strength_M,
                sigma_surface_nm2=sigma_atual,
                b_nm=b_nm_ppy,
                N=N_ppy,
                f=f_ppy,
                chains_adsorbed_nm2=chains_adsorbed_nm2,
                polymer_charge_sign=sinal_material
            )

            sigma_nova = res["Carga Resultante (cargas/nm²)"]
            espessura = res["Espessura da Camada (nm)"]
            regime = res["Regime"]
            adsorcao = res["Adsorção"]
            sigma_depositada = res["Carga do PPy (cargas/nm²)"]

        # ADSORÇÃO DE MoS2

        else:
            # Verifica se existe atração eletrostática
            if np.sign(sigma_atual) != sinal_material:

                # Contribuição efetiva de carga do MoS2
                sigma_depositada = (
                    sinal_material * sigma_mos2_layer_nm2
                )

                sigma_nova = sigma_atual + sigma_depositada
                espessura = espessura_mos2_nm
                regime = "Adsorção de nanosheet (simplificada)"
                adsorcao = True
            else:

                sigma_depositada = 0.0
                sigma_nova = sigma_atual
                espessura = 0.0
                regime = "Sem atração eletrostática"
                adsorcao = False

        # Armazenamento dos resultados

        resultados.append({

            "Camada": i + 1,
            "Material": material,
            "Adsorção": adsorcao,
            "Carga Antes (cargas/nm²)": sigma_antes,
            "Carga Depositada (cargas/nm²)": sigma_depositada,
            "Carga Após (cargas/nm²)": sigma_nova,
            "Espessura (nm)": espessura,
            "Regime": regime
        })

        # atualiza a superfície
        sigma_atual = sigma_nova

    return resultados


# (im)Prints e execução

if __name__ == "__main__":

    print("=" * 75)
    print("MODELO DE ADSORÇÃO E INVERSÃO DE CARGA")
    print("Sistema: Layer-by-Layer de PPy / MoS2")
    print("=" * 75)

    # PARÂMETROS DO SISTEMA
    params = {
        "T": 298.15,
        "epsilon_r": 78.5,
        "ionic_strength_M": 0.01,
        "sigma_inicial_nm2": -0.1,
        "b_nm_ppy": 0.7,
        "N_ppy": 100,
        "f_ppy": 0.3,
        "chains_adsorbed_nm2": 0.01,
        "sigma_mos2_layer_nm2": 0.3,
        "espessura_mos2_nm": 1.0
    }

    print("\nParâmetros utilizados:")
    for k, v in params.items():
        print(f"  {k}: {v}")

    # CAMADA ÚNICA: PPy SOBRE MoS2
    print("\n" + "-" * 75)
    print("CAMADA ÚNICA: PPy sobre MoS2")
    print("-" * 75)

    resultado_unico = joanny_adsorption_layer(
        T=params["T"],
        epsilon_r=params["epsilon_r"],
        ionic_strength_M=params["ionic_strength_M"],
        sigma_surface_nm2=params["sigma_inicial_nm2"],
        b_nm=params["b_nm_ppy"],
        N=params["N_ppy"],
        f=params["f_ppy"],
        chains_adsorbed_nm2=params["chains_adsorbed_nm2"],
        polymer_charge_sign=+1
    )

    for chave, valor in resultado_unico.items():
        if isinstance(valor, float):
            print(f"  {chave}: {valor:.4f}")
        else:
            print(f"  {chave}: {valor}")

    # SIMULAÇÃO LAYER-BY-LAYER
    print("\n" + "-" * 75)
    print("SIMULAÇÃO LAYER-BY-LAYER (5 CAMADAS)")
    print("-" * 75)

    camadas = simular_multicamadas_lbl(
        n_camadas=5,
        **params
    )

    print(
        f"\n{'Camada':<8}"
        f"{'Material':<10}"
        f"{'Adsorção':<12}"
        f"{'Carga Antes':<14}"
        f"{'Carga Depositada':<18}"
        f"{'Carga Após':<14}"
        f"{'Espessura':<12}"
    )

    print("-" * 95)

    for c in camadas:
        print(
            f"{c['Camada']:<8}"
            f"{c['Material']:<10}"
            f"{str(c['Adsorção']):<12}"
            f"{c['Carga Antes (cargas/nm²)']:<14.4f}"
            f"{c['Carga Depositada (cargas/nm²)']:<18.4f}"
            f"{c['Carga Após (cargas/nm²)']:<14.4f}"
            f"{c['Espessura (nm)']:<12.2f}"
        )

    # ANÁLISE DO REGIME
    print("\n" + "-" * 75)

    print("ANÁLISE DA ADSORÇÃO")
    print("-" * 75)
    print("Regime:", resultado_unico["Regime"])
    print(
        "kappa * delta:",
        f"{resultado_unico['kappa * delta']:.4f}"
    )
    print(
        "Inversão de carga:",
        resultado_unico["Inversão de Carga"]
    )
    print(
        "Carga resultante:",
        f"{resultado_unico['Carga Resultante (cargas/nm²)']:.4f}",
        "cargas/nm²"
    )

    # ESPESSURA TOTAL DO FILME
    espessura_total = sum(
        c["Espessura (nm)"] for c in camadas
    )
    print("\nEspessura total estimada do filme:")
    print(f"{espessura_total:.3f} nm")
    print("\n" + "=" * 75)
    print("FIM DA SIMULAÇÃO")
    print("=" * 75)